In [ ]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

import os

# ---------------------------------------------------------------------------
# Configuracion externalizada (widgets de Databricks / variables de entorno).
# ---------------------------------------------------------------------------
dbutils.widgets.text("catalog", "job_offers")
dbutils.widgets.text("silver_schema", "silver")
dbutils.widgets.text("gold_schema", "gold")


def _cfg(name: str, default: str = "") -> str:
    """Resuelve un parametro: widget -> variable de entorno -> default."""
    value = dbutils.widgets.get(name)
    return value if value else os.environ.get(name.upper(), default)


CATALOG = _cfg("catalog", "job_offers")
SILVER_SCHEMA = _cfg("silver_schema", "silver")
GOLD_SCHEMA = _cfg("gold_schema", "gold")


def silver_table(name: str) -> str:
    return f"{CATALOG}.{SILVER_SCHEMA}.{name}"


In [ ]:
df_linkedin=spark.read.table(silver_table("offers_linkedin"))
df_infojobs=spark.read.table(silver_table("offers_infojobs"))
df_indeed=spark.read.table(silver_table("offers_indeed"))
df_multisite=spark.read.table(silver_table("offers_multi_site"))

In [ ]:
display(df_linkedin)

In [ ]:
from functools import reduce
from pyspark.sql import DataFrame

# Orden de columnas puede variar, pero nombres iguales
result = reduce(
    lambda a, b: a.unionByName(b, allowMissingColumns=True),
    [df_linkedin, df_infojobs, df_indeed, df_multisite])

In [ ]:
from pyspark.sql import Window
from pyspark.sql import functions as F

UNICODE_DASHES = r"[\u2010\u2011\u2012\u2013\u2014\u2212\uFE58\uFE63\uFF0D]"

def _norm(c):
    c = F.trim(F.lower(c))
    c = F.regexp_replace(c, UNICODE_DASHES, "-")
    c = F.regexp_replace(c, r"\s*-\s*", "-")
    return F.trim(F.regexp_replace(c, r"\s+", " "))

def _keep_best(df, partition_cols):
    w = Window.partitionBy(*partition_cols).orderBy(
        F.when(F.col("description_clean").isNotNull() & (F.trim(F.col("description_clean")) != ""), 1).otherwise(0).desc(),
        F.when(F.col("salary_min_annual").isNotNull(), 1).otherwise(0).desc(),
        F.col("posted_date").desc_nulls_last())
    return (df.withColumn("_rn", F.row_number().over(w))
              .filter(F.col("_rn") == 1)
              .drop("_rn"))

result = (result
          .withColumn("title_norm", _norm(F.col("title")))
          .withColumn("company_norm", _norm(F.col("company_name"))))

result = _keep_best(result, ["job_id"])

w_fill = Window.partitionBy("title_norm", "company_norm")
result = result.withColumn(
    "salary_filled",
    F.coalesce(F.col("salary_min_annual"),
               F.first(F.col("salary_min_annual"), ignorenulls=True).over(w_fill)))

result = _keep_best(result, ["title_norm", "company_norm", "salary_filled"])
result = result.drop("title_norm", "company_norm", "salary_filled")

In [ ]:
import Prepare_Gold as dg

gold_tables = dg.build_gold(spark, result)

dg.write_gold(gold_tables, catalog=CATALOG, schema=GOLD_SCHEMA)



Dim_Companies

In [ ]:
companies_df_linkedin=spark.read.table(silver_table("companies_linkedin"))
companies_df_indeed=spark.read.table(silver_table("companies_indeed"))
companies_df_infojobs=spark.read.table(silver_table("companies_infojobs"))
companies_df_multi_site=spark.read.table(silver_table("companies_multi_site"))

In [ ]:
from pyspark.sql import functions as F

companies_df_linkedin = companies_df_linkedin.select(
    F.trim("company_name").alias("company_name"),
    F.col("company_industry"),
    F.col("company_description"),
    F.col("company_size"),
    F.lit("linkedin").alias("source"),
)

companies_df_indeed = companies_df_indeed.select(
    F.trim("company_name").alias("company_name"),      # Indeed usa 'company'
    F.col("company_rating"),
    F.col("company_review_count"),
    F.lit("indeed").alias("source"),
)
#2. Union (las columnas que falten en un lado se rellenan con NULL)
companies_df = companies_df_linkedin.unionByName(companies_df_indeed, allowMissingColumns=True)
#3. Añadir los 2 dataframes que solo tienen company_name
only_names = companies_df_multi_site.select(F.col("company_name")) \
    .union(companies_df_infojobs.select(F.col("company_name")))
#Al unir con allowMissingColumns=True, las columnas extra de companies quedan a NULL para estas filas:
companies_final = companies_df.unionByName(only_names, allowMissingColumns=True)
#4. Normalizar y deduplicar (imprescindible: la misma empresa aparece en varias fuentes)
from pyspark.sql import Window

w = Window.partitionBy("_key").orderBy(
    F.when(F.col("company_industry").isNotNull(), 1).otherwise(0).desc(),
    F.when(F.col("company_size").isNotNull(), 1).otherwise(0).desc(),
    F.when(F.col("company_rating").isNotNull(), 1).otherwise(0).desc(),
    F.col("source").asc(),
)
companies_final = (
    companies_final
    .withColumn("_key", F.lower(F.trim(F.regexp_replace("company_name", r"\s+", " "))))
    .withColumn("_rn", F.row_number().over(w))
    .filter(F.col("_rn") == 1)
    .drop("_rn", "_key")
)
companies_final = companies_final.filter(
    ~(F.col("company_name").rlike(r"^\d+[,.]\d{1,2}$"))
    & (F.trim(F.col("company_name")) != "")
)

In [ ]:
companies_final.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.companies_complete_catalog")